# Orchestration Frameworks

**Module:** 14 — AI Orchestration

LangGraph and peers — how to select an orchestration stack.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Map framework concepts to runtimes you already know
- Sketch a LangGraph-style graph in Python
- Use a selection matrix for build-vs-buy decisions


## LangGraph

### Definition
A graph-centric runtime for agents/workflows with explicit state, nodes, edges, and checkpointers.

### Why it matters
It aligns well with production needs: durability, cycles, HITL, streaming.

### How it works
Define `StateGraph` → add nodes → add edges/conditions → compile with checkpointer → invoke with `thread_id`.

### Intuition
BPM for LLMs — with cycles allowed when you need agent loops.

### Pitfalls
- Overusing cycles when a DAG suffices
- Giant untyped state
- No timeout policies

### When to use
Agentic apps that need resume and branching.


```mermaid
flowchart LR
  START --> retrieve
  retrieve --> draft
  draft --> review{human?}
  review -->|edit| draft
  review -->|ok| send
  send --> END
```

### Other names you'll hear
| Framework / system | Notes |
|--------------------|-------|
| Temporal / durable exec | General workflows; AI steps as activities |
| Prefect / Airflow | Data/ML pipelines; coarser-grained |
| CrewAI / AutoGen | Multi-agent collaboration patterns |
| Semantic Kernel | Skills/plans in .NET/Python ecosystems |
| Custom | Often wins under strict compliance |


In [ ]:
# Demo 1: LangGraph-like graph dict
graph = {
    "nodes": ["retrieve", "draft", "human_review", "send"],
    "edges": [
        ("__start__", "retrieve"),
        ("retrieve", "draft"),
        ("draft", "human_review"),
        ("human_review", "send"),  # simplified
        ("send", "__end__"),
    ],
}

def next_node(current, approved=True):
    if current == "draft" and not approved:
        return "draft"
    for a, b in graph["edges"]:
        if a == current:
            return b
    return None

cur = "__start__"
path = []
for _ in range(10):
    cur = next_node(cur) if cur != "draft" else next_node("draft", True)
    if cur in (None, "__end__"):
        path.append("__end__"); break
    path.append(cur)
print(path)


In [ ]:
# Demo 2: conditional edge function
def route_after_draft(state):
    if state.get("risk") == "high":
        return "human_review"
    if state.get("confidence", 0) < 0.5:
        return "retrieve"  # loop for more evidence
    return "send"

for s in [{"risk": "high"}, {"confidence": 0.2}, {"confidence": 0.9}]:
    print(s, "->", route_after_draft(s))


In [ ]:
# Demo 3: placeholder remote invoke shape
import os, json
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", "YOUR_LANGSMITH_API_KEY")
invoke_body = {
    "input": {"messages": [{"role": "user", "content": "Draft a refund email"}]},
    "config": {"configurable": {"thread_id": "thr_123"}},
}
invoke_response = {
    "values": {"messages": [{"role": "assistant", "content": "...draft..."}], "status": "waiting_human"},
}
print(json.dumps({"req": invoke_body, "resp": invoke_response}, indent=2))
print("tracing key placeholder?", LANGSMITH_API_KEY.startswith("YOUR_"))


## Selection Matrix

| Need | Lean toward |
|------|-------------|
| Fast agent prototype | Lightweight agent SDK / LangGraph quickstart |
| Multi-day durable jobs | Temporal-style durable execution |
| Heavy data DAG | Airflow/Prefect + LLM tasks |
| Strict multi-tenant SaaS | Custom runtime + Postgres |
| Enterprise .NET shop | Semantic Kernel / Azure orchestration |
| Multi-agent role play | Crew/AutoGen — still wrap with state/ops |

### Questions to ask vendors/frameworks
1. How are checkpoints stored and encrypted?  
2. How do we version graphs?  
3. What's the HITL story?  
4. Can we export traces to our observability stack?  
5. What's the failure model under partial tool success?  


In [ ]:
# Demo 4: score frameworks for a scenario
def score(scenario_weights, framework_caps):
    return sum(scenario_weights[k] * framework_caps.get(k, 0) for k in scenario_weights)

weights = {"durability": 0.4, "hitl": 0.3, "multi_agent": 0.1, "simplicity": 0.2}
frameworks = {
    "langgraph": {"durability": 0.8, "hitl": 0.8, "multi_agent": 0.6, "simplicity": 0.5},
    "temporal+llm": {"durability": 0.95, "hitl": 0.7, "multi_agent": 0.4, "simplicity": 0.3},
    "crew_only": {"durability": 0.3, "hitl": 0.3, "multi_agent": 0.9, "simplicity": 0.7},
}
print(sorted(((name, round(score(weights, caps), 3)) for name, caps in frameworks.items()), key=lambda x: -x[1]))


### Try it yourself — Frameworks

1. Re-score with weights for a HIPAA async document pipeline.
2. Map LangGraph terms to Temporal (workflow/activity/signal).
3. List 3 reasons you might still build custom.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `node` | Unit of work in a graph |
| `conditional edge` | Branch based on state |
| `compile` | Build executable graph |
| `thread_id` | Durable conversation/run key |


## Concrete Mapping Table

| Concept | LangGraph | Temporal | Airflow |
|---------|-----------|----------|---------|
| Unit | node | activity | task |
| Durable progress | checkpointer | workflow history | DB task instances |
| Human wait | interrupt/wait | signal+wait | sensor |
| Versioning | app-level | workflow types | DAG versioning |
| Best fit | AI agents | mixed long-running biz | data/ML batch |

### Exit strategy
Export: state JSON, prompt templates, tool schemas, traces. Avoid proprietary step logic without adapters.


In [ ]:
# Minimal portable IR for a graph
ir = {
    "name": "support_v3",
    "nodes": [
        {"id": "retrieve", "type": "tool", "tool": "search_kb"},
        {"id": "draft", "type": "llm", "model": "gpt-4o-mini"},
        {"id": "human", "type": "hitl"},
        {"id": "send", "type": "tool", "tool": "send_email"},
    ],
    "edges": [["retrieve", "draft"], ["draft", "human"], ["human", "send"]],
}
print(ir["name"], "nodes", len(ir["nodes"]))


### Try it yourself — Frameworks deepen

1. Translate the IR above into pseudo-Temporal activities/signals.
2. List data-residency questions for a hosted agent cloud.


## Build-vs-buy red flags
- Cannot export runs
- No HIPAA/BAA when you need it
- Hidden prompt changes without version pins
- No per-tenant authz model


## Key Takeaways

- Frameworks encode state+control patterns
- Score against your real constraints
- Durable execution and HITL matter more than demos
- Keep an exit strategy / exportable state
